# Polimorfismo y diseño SOLID

Llegamos al último vídeo del bloque. Ya sabemos crear clases, relacionarlas con
herencia y composición, y definir contratos con clases abstractas. Ahora juntamos
todo eso en dos ideas que separan el código que **crece bien** del que se convierte
en un nudo imposible de mantener:

- **Polimorfismo**: tratar objetos distintos a través de una **interfaz común**, sin
  preguntar por su tipo.
- **SOLID**: cinco principios de diseño, con foco en los dos más prácticos al
  empezar: **OCP** (abierto/cerrado) y **DIP** (inversión de dependencias).

Usaremos un ejemplo muy reconocible: **métodos de pago** (PayPal, tarjeta, Bizum...).

> 📚 Documentación oficial de referencia:
> **[abc](https://docs.python.org/es/3/library/abc.html)** (la base de nuestras
> interfaces).

---

## 1. El problema: un `if/elif` por cada tipo

Imagina que tenemos que procesar un pago según el método elegido. Sin polimorfismo, la
solución típica es una cadena de `if/elif` que mira el tipo:

In [ ]:
def procesar_pago(metodo, cantidad):
    if metodo == "paypal":
        return f"Pagados {cantidad}€ con PayPal"
    elif metodo == "tarjeta":
        return f"Pagados {cantidad}€ con tarjeta"
    elif metodo == "bizum":
        return f"Pagados {cantidad}€ con Bizum"
    else:
        raise ValueError(f"Método de pago desconocido: {metodo}")


print(procesar_pago("paypal", 50))
print(procesar_pago("bizum", 30))

Funciona, pero **no escala bien**: cada vez que aparezca un método de pago nuevo hay
que **volver a tocar** esta función y añadir otro `elif`. Y si esa lógica aparece en
varios sitios, hay que actualizarla en todos. Es justo lo que el polimorfismo evita.

---

## 2. Polimorfismo: una interfaz común

La idea del polimorfismo es: en lugar de preguntar "¿de qué tipo eres?", le pedimos a
cada objeto que **haga la acción él mismo**. Todos los métodos de pago comparten una
misma operación, `pagar`, aunque cada uno la implemente a su manera.

Definimos primero la **abstracción** con una ABC (como en el vídeo anterior): un
contrato que dice "todo método de pago sabe `pagar`".

In [ ]:
from abc import ABC, abstractmethod


class MetodoPago(ABC):          # la abstracción: el contrato común
    @abstractmethod
    def pagar(self, cantidad):
        ...


class PayPal(MetodoPago):
    def pagar(self, cantidad):
        return f"Pagados {cantidad}€ con PayPal"


class Tarjeta(MetodoPago):
    def pagar(self, cantidad):
        return f"Pagados {cantidad}€ con tarjeta"


class Bizum(MetodoPago):
    def pagar(self, cantidad):
        return f"Pagados {cantidad}€ con Bizum"


def procesar_pago(metodo_pago, cantidad):
    # polimorfismo: no nos importa el tipo concreto, solo que sabe pagar()
    return metodo_pago.pagar(cantidad)

In [ ]:
# La misma función sirve para todos, sin un solo if por tipo
for metodo in [PayPal(), Tarjeta(), Bizum()]:
    print(procesar_pago(metodo, 50))

Fíjate: `procesar_pago` **no tiene ningún `if` por tipo**. Le da igual si recibe un
`PayPal` o un `Bizum`; solo confía en que sabrá responder a `pagar`. Eso es el
polimorfismo: *mismo mensaje, distintos objetos*.

---

## 3. OCP: abierto a extensión, cerrado a modificación

El primer principio SOLID que se ve aquí es el **Open/Closed Principle**: el código
debe estar **abierto a extensión** pero **cerrado a modificación**. Es decir,
deberíamos poder añadir comportamiento nuevo **sin tocar** el código que ya funciona.

Con el `if/elif` del principio, añadir un método nuevo obligaba a modificar
`procesar_pago`. Con polimorfismo, basta con **crear una clase nueva**:

In [ ]:
class Cripto(MetodoPago):   # método nuevo... y NO tocamos procesar_pago
    def pagar(self, cantidad):
        return f"Pagados {cantidad}€ con criptomoneda"


print(procesar_pago(Cripto(), 100))  # funciona sin cambiar nada del código anterior

> 💡 Añadir `Cripto` no nos ha obligado a editar `procesar_pago` ni las clases
> existentes. El sistema está **cerrado a modificación** (no rompemos lo que ya
> funciona) pero **abierto a extensión** (podemos seguir añadiendo métodos de pago).

---

## 4. DIP: depender de abstracciones, no de concreciones

El segundo principio clave es el **Dependency Inversion Principle**: las partes
importantes del código deben depender de **abstracciones**, no de implementaciones
concretas.

Nuestra función `procesar_pago` no depende de `PayPal` ni de `Bizum` (las
concreciones), sino del contrato `MetodoPago` (la abstracción). Eso es justo lo que
le permite funcionar con cualquier método de pago, presente o futuro.

In [ ]:
# Cualquier objeto que CUMPLA el contrato MetodoPago sirve.
# El código de alto nivel (procesar_pago) no conoce los detalles concretos.
metodos = [PayPal(), Tarjeta(), Bizum(), Cripto()]
total = 0
for metodo in metodos:
    print(procesar_pago(metodo, 25))
    total += 25
print("Total procesado:", total, "€")

> 💡 "Inversión" se refiere a esto: en vez de que el código de alto nivel dependa de
> los detalles de bajo nivel, **ambos dependen de una abstracción** común
> (`MetodoPago`). Cambiar o añadir implementaciones concretas ya no afecta a quien
> las usa.

---

## 5. Los cinco principios SOLID de un vistazo

SOLID son cinco principios de diseño orientado a objetos. Ya hemos visto en
profundidad OCP y DIP; estos son los cinco, resumidos:

| Principio | Idea en una frase | Ejemplo en nuestro código |
|-----------|-------------------|----------------------------|
| **S**RP — *Single Responsibility* | Una clase, una sola razón para cambiar | Separar "procesar el pago" de "enviar el recibo por email" |
| **O**CP — *Open/Closed* | Abierto a extensión, cerrado a modificación | Añadir `Cripto(MetodoPago)` sin tocar `procesar_pago` |
| **L**SP — *Liskov Substitution* | Una subclase debe poder sustituir a su base sin romper nada | `Bizum` funciona en cualquier sitio donde se espere un `MetodoPago` |
| **I**SP — *Interface Segregation* | Interfaces pequeñas y enfocadas | No obligar a un método de pago a implementar cosas que no usa |
| **D**IP — *Dependency Inversion* | Depender de abstracciones, no de concreciones | `procesar_pago` depende de `MetodoPago`, no de `PayPal` |

> 💡 No hace falta aplicarlos todos a la vez ni obsesionarse con ellos. Son una
> **guía** para reconocer cuándo un diseño se está volviendo rígido y cómo aflojarlo.

---

## 6. Resumen

En este vídeo hemos cerrado el bloque con las ideas de diseño que dan sentido a todo
lo anterior:

1. El **polimorfismo** consiste en tratar objetos distintos a través de una
   **interfaz común** (`pagar`), sin preguntar por su tipo. Adiós a los `if/elif` por
   tipo.
2. **OCP**: con polimorfismo, añadir comportamiento nuevo es **crear una clase**, no
   modificar el código existente.
3. **DIP**: el código importante depende de la **abstracción** (`MetodoPago`), no de
   las implementaciones concretas.
4. **SOLID** (SRP, OCP, LSP, ISP, DIP) es una guía para que el código **crezca sin
   volverse frágil**.

Y con esto cerramos el recorrido completo del bloque de POO: desde *qué es una clase*
(vídeo 1) hasta *diseñar sistemas flexibles* (vídeo 9). La pieza que une todo es
siempre la misma: clases que colaboran a través de interfaces claras, en lugar de
código que pregunta constantemente "¿de qué tipo eres?".